In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

from pathlib import Path
import os
import sys

In [2]:
# Get current file location
try:
    BASE_DIR = Path(__file__).resolve().parents[2]
except NameError:
    BASE_DIR = Path.cwd().parents[1]

# add src to python path
sys.path.append(str(BASE_DIR))

# Path to data folder
DATA_DIR = BASE_DIR / "data"

# load data
data_train = pd.read_csv(DATA_DIR / "train_FD001.txt", sep=r"\s+", header=None)
data_test = pd.read_csv(DATA_DIR / "test_FD001.txt", sep=r"\s+", header=None)
data_rul = pd.read_csv(DATA_DIR / "RUL_FD001.txt", header=None)

# prepare name for each columns
cols = (
    ['engine_id','cycle'] +
    [f'setting_{i}' for i in range(1,4)] +
    [f'sensor_{i}' for i in range(1,22)]
)

In [3]:
data_train.head()

,0,1,2,3,4,5,6,7,8,9,...,16,17,18,19,20,21,22,23,24,25
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


In [4]:
from src.data.preprocess import preprocess

train_pre, test_pre = preprocess(data_train, data_test, cols, clip=False)

In [5]:
train_pre.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_1,sensor_2,sensor_3,sensor_4,sensor_5,...,sensor_13,sensor_14,sensor_15,sensor_16,sensor_17,sensor_18,sensor_19,sensor_20,sensor_21,RUL
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191.0
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190.0
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189.0
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188.0
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187.0


In [6]:
from src.features.build_features import build_features

train_df, test_df = build_features(train_pre, test_pre)

sensors with low variance:
sensor_1     0.000000e+00
sensor_5     3.155597e-30
sensor_6     1.929279e-06
sensor_10    0.000000e+00
sensor_16    1.926023e-34
sensor_18    0.000000e+00
sensor_19    0.000000e+00
dtype: float64


In [7]:
train_df.head()

,engine_id,cycle,setting_1,setting_2,setting_3,sensor_2,sensor_3,sensor_4,sensor_7,sensor_8,...,sensor_8_diff,sensor_9_diff,sensor_11_diff,sensor_12_diff,sensor_13_diff,sensor_14_diff,sensor_15_diff,sensor_17_diff,sensor_20_diff,sensor_21_diff
9,1,10,-0.0033,0.0001,100.0,-1.941707,0.116927,-0.941491,0.251154,-0.657216,...,0.000000,0.237294,-0.973488,0.000000,0.139049,0.193440,1.487836,0.645692,-0.553275,0.580148
10,1,11,0.0018,-0.0003,100.0,-0.801801,-1.430944,-0.921492,1.324514,-0.657216,...,0.000000,-0.094646,0.449302,-0.528788,-0.695244,0.587134,0.143984,-0.645692,-0.055327,0.085914
11,1,12,0.0016,0.0002,100.0,-1.241765,-1.160189,-0.975934,1.301917,-0.093707,...,0.563509,-0.010868,0.112326,0.542347,0.139049,-0.331836,-1.071882,-0.645692,0.663930,-1.041125
12,1,13,-0.0019,0.0004,100.0,0.778069,-1.359178,-0.900382,0.081676,0.328925,...,0.422632,-0.115477,0.748837,0.067793,0.834293,-0.322399,0.570604,1.291384,-0.719257,-0.834194
13,1,14,0.0009,-0.0000,100.0,-0.661813,0.395838,-1.085929,1.256723,-0.093707,...,-0.422632,0.024907,0.224651,-0.244056,-1.112391,0.331836,-0.501278,0.000000,1.383187,0.987545


In [8]:
# train-test-split
X = train_df.drop(columns=["RUL"])
y = train_df["RUL"]

groups = train_df["engine_id"]

split = GroupShuffleSplit(test_size=0.2, n_splits=1)

train_idx, val_idx = next(split.split(X, y, groups))

X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

In [9]:
# dataset for deep learning
X_train_scaled = train_df.iloc[train_idx].copy()
X_val_scaled = train_df.iloc[val_idx].copy()

# scaled data
scaler = StandardScaler()

sensor_cols = [c for c in train_df.columns if "sensor" in c]

X_train_scaled[sensor_cols] = scaler.fit_transform(X_train[sensor_cols])
X_val_scaled[sensor_cols] = scaler.transform(X_val[sensor_cols])

# Reshape into sequences
from src.features.build_features import create_sequences

X_train_seq, y_train_seq = create_sequences(X_train_scaled)
X_val_seq, y_val_seq = create_sequences(X_val_scaled)

# dataloader
from torch.utils.data import DataLoader
from src.model.dataset import RULDataset

train_dataset = RULDataset(X_train_seq, y_train_seq)
val_dataset = RULDataset(X_val_seq, y_val_seq)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64)

In [10]:
print(np.isnan(X_train_seq).sum())
print(np.isnan(y_train_seq).sum())
print(np.isnan(X_val_seq).sum())
print(np.isnan(y_val_seq).sum())

0
0
0
0


In [11]:
data = {

    # classical ML
    "ml": {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val
    },

    # deep learning
    "dl": {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "X_train_seq": X_train_seq,
        "X_val_seq": X_val_seq
    }
}

In [12]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(device)

cpu


In [13]:
config = {

    "training": {
        "epochs": 20,
        "batch_size": 64,
        "lr": 1e-3,
        "device": device
    },

    "data": {
        "input_size": X_train_seq.shape[2],   # number of sensors/features
        "window_size": X_train_seq.shape[1]
    },

    "models": {

        "xgboost": {
            "objective":"reg:squarederror",
            "n_estimators": 500,
            "max_depth": 6,
            "learning_rate": 0.05,
            "subsample": 0.8,
            "colsample_bytree": 0.8,
            "random_state": 42
        },

        "lightgbm": {
            "n_estimators": 500,
            "num_leaves": 31,
            "learning_rate": 0.05
        },

        "lstm": {
            "hidden_size": 64,
            "num_layers": 2,
            "dropout": 0.2
        },

        "tcn": {
            "num_channels": 64,
            "kernel_size": 3
        },

        "transformer": {
            "d_model": 64,
            "nhead": 4,
            "num_layers": 2
        }
    }
}

In [14]:
from src.model.registry import MODEL_REGISTRY
from src.model.evaluate import evaluate_model

results = {}

for name, train_fn in MODEL_REGISTRY.items():
    
    print(f"start f{name}")
    model = train_fn(config, data)

    rmse = evaluate_model(name, model, data, device)

    results[name] = rmse

import pandas as pd

results_df = pd.DataFrame.from_dict(
    results,
    orient="index",
    columns=["RMSE"]
)

print(results_df)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001230 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 13041
[LightGBM] [Info] Number of data points in the train set: 15319, number of used features: 60
[LightGBM] [Info] Start training from score 99.561655


/opt/anaconda3/envs/nasa_rul/lib/python3.12/site-packages/torch/nn/modules/transformer.py:385: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(


                  RMSE
xgboost      51.615319
lightgbm     52.706686
lstm         46.335109
tcn          42.762890
transformer  80.635832
